# Make ML models for prediction gte parameters

# Parameters

In [3]:
from config import parameters as gtep

In [6]:
for alias, parameter in gtep.items():
    print(f"{alias:<15}: {parameter}")

l              : thermal_conductivity
hc             : heat_capacity
Cp             : heat_capacity_pressure
Cv             : heat_capacity_volume
k              : adiabatic_index
gc             : gas_const
eo             : excess_oxidizing
T              : static_temperature
P              : static_pressure
D              : staticdensity
TT             : total_temperature
PP             : total_pressure
DD             : total_density
m              : mass
v              : volume
a              : sound_speed
a_critical     : critical_sound_speed
c              : absolute_velocity
u              : portable_velocity
w              : relative_velocity
mf             : mass_flow
pipi           : total_pressure_ratio
pi             : static_pressure_ratio
titi           : total_temperature_ratio
ti             : static_temperature_ratio
mach           : mach_number
Nu             : nusselt_number
efficiency     : efficiency
effeff         : total_efficiency
peff           : pressure_efficie

# Generate data

In [31]:
import time
import numpy as np
import pandas as pd

from thermodynamics import gas_const, heat_capacity_p
from substance import Substance

## Compressor

In [29]:
from gte import Compressor

In [30]:
air = Substance(
    "air",
    composition={"N2": 0.78, "O2": 0.21, "Ar": 0.009, "CO2": 0.0004},
    parameters={
        gtep.mf: 50.0,
        gtep.gc: 287.14,
        gtep.TT: 300.0,
        gtep.PP: 101325.0,
        gtep.Cp: 1006.0,
        gtep.k: 1.4,
        gtep.c: 0.0,
    },
    functions={
        gtep.gc: lambda temperature: gas_const("air"),
        gtep.Cp: lambda temperature: heat_capacity_p("air", temperature),
    },
)

In [20]:
data = []

In [21]:
for pipi in range(4, 12, 1):
    for effeff in range(85, 95, 1):
        compressor = Compressor("generator")
        setattr(compressor, gtep.pipi, pipi)
        setattr(compressor, gtep.effeff, effeff / 100)

        compressor.calculate(air)

        data.append(compressor.summary)

--------------------
total_pressure_ratio: 4
total_efficiency: 0.85
power: 8606001.717811026
mass_flow_inlet: 50.0
gas_const_inlet: 287.14
total_temperature_inlet: 300.0
total_pressure_inlet: 101325.0
heat_capacity_pressure_inlet: 1006.0
adiabatic_index_inlet: 1.4
absolute_velocity_inlet: 0.0
mass_flow_outlet: 50.0
total_temperature_outlet: 470.0249781484548
total_pressure_outlet: 405300.0
gas_const_outlet: 287.14
heat_capacity_pressure_outlet: 1025.0383159977407
total_density_outlet: 3.00304589708258
adiabatic_index_outlet: 1.3891322066669132
critical_sound_speed_outlet: 396.1630928926236
--------------------
--------------------
total_pressure_ratio: 4
total_efficiency: 0.86
power: 8506187.700602202
mass_flow_inlet: 50.0
gas_const_inlet: 287.14
total_temperature_inlet: 300.0
total_pressure_inlet: 101325.0
heat_capacity_pressure_inlet: 1006.0
adiabatic_index_inlet: 1.4
absolute_velocity_inlet: 0.0
mass_flow_outlet: 50.0
total_temperature_outlet: 468.0771215803783
total_pressure_outlet

In [77]:
df = pd.DataFrame(data)
df

,total_pressure_ratio,total_efficiency,power,mass_flow_inlet,gas_const_inlet,total_temperature_inlet,total_pressure_inlet,heat_capacity_pressure_inlet,adiabatic_index_inlet,absolute_velocity_inlet,mass_flow_outlet,total_temperature_outlet,total_pressure_outlet,gas_const_outlet,heat_capacity_pressure_outlet,total_density_outlet,adiabatic_index_outlet,critical_sound_speed_outlet
0,4,0.85,8.606002e+06,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,470.024978,405300.0,287.14,1025.038316,3.003046,1.389132,396.163093
1,4,0.86,8.506188e+06,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,468.077122,405300.0,287.14,1024.682045,3.015543,1.389320,395.352555
2,4,0.87,8.408661e+06,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,466.173256,405300.0,287.14,1024.335552,3.027858,1.389503,394.558577
3,4,0.88,8.313345e+06,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,464.311910,405300.0,287.14,1023.998472,3.039996,1.389681,393.780653
4,4,0.89,8.220164e+06,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,462.491680,405300.0,287.14,1023.670456,3.051961,1.389855,393.018299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,11,0.90,1.635790e+07,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,619.088201,1114575.0,287.14,1056.181427,6.269936,1.373374,453.575328
76,11,0.91,1.618006e+07,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,615.719384,1114575.0,287.14,1055.420228,6.304241,1.373744,452.365226
77,11,0.92,1.600605e+07,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,612.420605,1114575.0,287.14,1054.676373,6.338198,1.374106,451.176854
78,11,0.93,1.583573e+07,50.0,287.14,300.0,101325.0,1006.0,1.4,0.0,50.0,609.189705,1114575.0,287.14,1053.949342,6.371813,1.374461,450.009619


In [81]:
now = time.strftime("%Y-%m-%d-%H-%M-%S", time.localtime())
df.to_excel(f"data/compressor_{now}.xlsx", index=False, header=True)

# ML

In [118]:
import os
import pickle

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [98]:
MODELS = (LinearRegression, DecisionTreeRegressor, RandomForestRegressor)
METRICS = (mean_absolute_error, mean_squared_error)

In [99]:
os.listdir("data")

['compressor_2025-11-02-18-04-09.xlsx']

In [84]:
df = pd.read_excel("data/compressor_2025-11-02-18-04-09.xlsx")
df

,total_pressure_ratio,total_efficiency,power,mass_flow_inlet,gas_const_inlet,total_temperature_inlet,total_pressure_inlet,heat_capacity_pressure_inlet,adiabatic_index_inlet,absolute_velocity_inlet,mass_flow_outlet,total_temperature_outlet,total_pressure_outlet,gas_const_outlet,heat_capacity_pressure_outlet,total_density_outlet,adiabatic_index_outlet,critical_sound_speed_outlet
0,4,0.85,8.606002e+06,50,287.14,300,101325,1006,1.4,0,50,470.024978,405300,287.14,1025.038316,3.003046,1.389132,396.163093
1,4,0.86,8.506188e+06,50,287.14,300,101325,1006,1.4,0,50,468.077122,405300,287.14,1024.682045,3.015543,1.389320,395.352555
2,4,0.87,8.408661e+06,50,287.14,300,101325,1006,1.4,0,50,466.173256,405300,287.14,1024.335552,3.027858,1.389503,394.558577
3,4,0.88,8.313345e+06,50,287.14,300,101325,1006,1.4,0,50,464.311910,405300,287.14,1023.998472,3.039996,1.389681,393.780653
4,4,0.89,8.220164e+06,50,287.14,300,101325,1006,1.4,0,50,462.491680,405300,287.14,1023.670456,3.051961,1.389855,393.018299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,11,0.90,1.635790e+07,50,287.14,300,101325,1006,1.4,0,50,619.088201,1114575,287.14,1056.181427,6.269936,1.373374,453.575328
76,11,0.91,1.618006e+07,50,287.14,300,101325,1006,1.4,0,50,615.719384,1114575,287.14,1055.420228,6.304241,1.373744,452.365226
77,11,0.92,1.600605e+07,50,287.14,300,101325,1006,1.4,0,50,612.420605,1114575,287.14,1054.676373,6.338198,1.374106,451.176854
78,11,0.93,1.583573e+07,50,287.14,300,101325,1006,1.4,0,50,609.189705,1114575,287.14,1053.949342,6.371813,1.374461,450.009619


In [119]:
targets = (gtep.pipi, gtep.effeff, gtep.power)

In [125]:
for target in targets:
    print(target)

    y = df[target]
    x = df.drop([target], axis=1)

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, shuffle=True)

    for Model in MODELS:
        print(f"\t{Model.__name__}")
        model = Model()
        model.fit(x_train, y_train)
        for metric in METRICS:
            print(f"\t\t{metric.__name__:<20}: {metric(y_test, model.predict(x_test)):.6f}")

        with open(f"models/compressor_{target}.pkl", "wb") as file:
            pickle.dump(model, file)

        print()

total_pressure_ratio
	LinearRegression
		mean_absolute_error : 0.000000
		mean_squared_error  : 0.000000

	DecisionTreeRegressor
		mean_absolute_error : 0.037037
		mean_squared_error  : 0.037037

	RandomForestRegressor
		mean_absolute_error : 0.138889
		mean_squared_error  : 0.066322

total_efficiency
	LinearRegression
		mean_absolute_error : 0.001602
		mean_squared_error  : 0.000005

	DecisionTreeRegressor
		mean_absolute_error : 0.011481
		mean_squared_error  : 0.000152

	RandomForestRegressor
		mean_absolute_error : 0.009307
		mean_squared_error  : 0.000159

power
	LinearRegression
		mean_absolute_error : 1.362309
		mean_squared_error  : 3.302799

	DecisionTreeRegressor
		mean_absolute_error : 143061.633921
		mean_squared_error  : 27067448643.289940

	RandomForestRegressor
		mean_absolute_error : 130114.758525
		mean_squared_error  : 32825159278.360275

